# Model Validation: Statistical Performance Metrics

---

## Overview

Before trusting any numerical model's predictions, it must be **validated** against observed (measured) data. This notebook implements six standard statistical metrics for comparing simulated vs. observed time series, commonly used in coastal and hydrological modelling:

| Metric | Abbreviation | Range | Perfect Score | What it measures |
|--------|-------------|-------|--------------|------------------|
| Bias (Mean Error) | Bias | $(-\infty, +\infty)$ | 0 | Systematic over/under-prediction |
| Mean Absolute Error | MAE | $[0, +\infty)$ | 0 | Average magnitude of errors |
| Root Mean Square Error | RMSE | $[0, +\infty)$ | 0 | Penalizes large errors more than MAE |
| Pearson Correlation | CC | $[-1, 1]$ | 1 | Linear agreement (phase/timing) |
| Nash-Sutcliffe Efficiency | NSE | $(-\infty, 1]$ | 1 | Skill relative to using the observed mean |
| Kling-Gupta Efficiency | KGE | $(-\infty, 1]$ | 1 | Decomposes into correlation, bias ratio, and variability ratio |

### Why Multiple Metrics?

No single metric tells the full story:
- **Bias** can be zero even if individual errors are large (cancellation)
- **CC** can be high even with a systematic offset
- **NSE** is sensitive to outliers and the observed variance
- **KGE** explicitly separates three components of model skill

Using all six provides a comprehensive assessment of model performance.

### Dependencies

```
numpy, scikit-learn
```

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

%matplotlib inline

## 1. Validation Metrics Function

### Mathematical Definitions

Given simulated values $s_i$ and observed values $o_i$ for $i = 1, \ldots, N$:

**Bias** (mean error):
$$\text{Bias} = \frac{1}{N} \sum_{i=1}^{N} (s_i - o_i)$$

**MAE** (mean absolute error):
$$\text{MAE} = \frac{1}{N} \sum_{i=1}^{N} |s_i - o_i|$$

**RMSE** (root mean square error):
$$\text{RMSE} = \sqrt{\frac{1}{N} \sum_{i=1}^{N} (s_i - o_i)^2}$$

**CC** (Pearson correlation coefficient):
$$r = \frac{\sum (s_i - \bar{s})(o_i - \bar{o})}{\sqrt{\sum (s_i - \bar{s})^2 \sum (o_i - \bar{o})^2}}$$

**NSE** (Nash-Sutcliffe Efficiency):
$$\text{NSE} = 1 - \frac{\sum (o_i - s_i)^2}{\sum (o_i - \bar{o})^2}$$

NSE = 1 is perfect; NSE = 0 means the model is no better than using $\bar{o}$; NSE < 0 means the model is worse than the mean.

**KGE** (Kling-Gupta Efficiency):
$$\text{KGE} = 1 - \sqrt{(r - 1)^2 + (\beta - 1)^2 + (\gamma - 1)^2}$$

where $\beta = \bar{s}/\bar{o}$ (bias ratio) and $\gamma = \sigma_s / \sigma_o$ (variability ratio). KGE decomposes model skill into three independent components.

In [ ]:
def calculate_validation_metrics(simulated_data, observed_data):
    """
    Calculates bias, MAE, RMSE, CC, NSE, and KGE for model validation.

    Parameters:
        simulated_data (np.array): Array of simulated values.
        observed_data (np.array): Array of observed values.

    Returns:
        tuple: (bias, mae, rmse, correlation_coefficient, nse, kge)
    """
    simulated_data = np.array(simulated_data)
    observed_data = np.array(observed_data)

    # Bias (Mean Error)
    bias = np.mean(simulated_data - observed_data)

    # Mean Absolute Error (MAE)
    mae = mean_absolute_error(observed_data, simulated_data)

    # Root Mean Square Error (RMSE)
    rmse = np.sqrt(mean_squared_error(observed_data, simulated_data))

    # Correlation Coefficient (CC) - Pearson's r
    correlation_coefficient = np.corrcoef(simulated_data, observed_data)[0, 1]

    # Nash-Sutcliffe Efficiency (NSE)
    nse = 1 - (np.sum((observed_data - simulated_data)**2) /
               np.sum((observed_data - np.mean(observed_data))**2))

    # Kling-Gupta Efficiency (KGE)
    r = correlation_coefficient
    beta = np.mean(simulated_data) / np.mean(observed_data)   # Bias ratio
    gamma = np.std(simulated_data) / np.std(observed_data)     # Variability ratio
    kge = 1 - np.sqrt((r - 1)**2 + (beta - 1)**2 + (gamma - 1)**2)

    return bias, mae, rmse, correlation_coefficient, nse, kge

## 2. Example: Synthetic Wave Height Data

We create synthetic observed and simulated wave height data to demonstrate the metrics:
- **Observed**: $H_s = 2 \sin(t) + \text{noise}$
- **Simulated**: $H_s = 2.1 \sin(t + 0.1) + \text{noise}$ (slight amplitude and phase bias)

This mimics a typical scenario where the model slightly overpredicts amplitude and has a small timing error.

In [ ]:
np.random.seed(42)

t = np.linspace(0, 10, 200)
observed = 2.0 * np.sin(t) + 0.5 * np.random.randn(len(t))
simulated = 2.1 * np.sin(t + 0.1) + 0.3 * np.random.randn(len(t))

bias, mae, rmse, cc, nse, kge = calculate_validation_metrics(simulated, observed)

print("=" * 40)
print("Model Validation Metrics")
print("=" * 40)
print(f"  Bias:  {bias:+.4f}")
print(f"  MAE:   {mae:.4f}")
print(f"  RMSE:  {rmse:.4f}")
print(f"  CC:    {cc:.4f}")
print(f"  NSE:   {nse:.4f}")
print(f"  KGE:   {kge:.4f}")

## 3. Visualization: Observed vs. Simulated

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Time series comparison
ax1.plot(t, observed, 'b-', lw=1, alpha=0.7, label='Observed')
ax1.plot(t, simulated, 'r-', lw=1, alpha=0.7, label='Simulated')
ax1.set_xlabel('Time')
ax1.set_ylabel('$H_s$ (m)')
ax1.set_title('Time Series Comparison')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Scatter plot
ax2.scatter(observed, simulated, s=15, alpha=0.5, c='steelblue', edgecolors='none')
lim = [min(observed.min(), simulated.min()) - 0.5,
       max(observed.max(), simulated.max()) + 0.5]
ax2.plot(lim, lim, 'k--', lw=1, label='1:1 line')
ax2.set_xlim(lim); ax2.set_ylim(lim)
ax2.set_xlabel('Observed $H_s$ (m)')
ax2.set_ylabel('Simulated $H_s$ (m)')
ax2.set_title(f'Scatter Plot (CC={cc:.3f}, RMSE={rmse:.3f})')
ax2.legend()
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Interpreting the Results

| Metric | Value | Interpretation |
|--------|-------|----------------|
| Bias | Small positive | Model slightly overpredicts on average |
| MAE | Low | Average error magnitude is acceptable |
| RMSE | Low | No major outlier errors |
| CC | High (~0.9+) | Strong linear agreement; timing is good |
| NSE | Close to 1 | Model explains most of the observed variance |
| KGE | Close to 1 | Balanced performance across correlation, bias, and variability |

### Guidelines for SWAN Validation

Typical acceptance criteria for nearshore wave models:

- **CC > 0.90** for significant wave height
- **RMSE < 20%** of the mean observed $H_s$
- **Bias < 10%** of the mean observed $H_s$
- **NSE > 0.70** indicates good model performance
- **KGE > 0.50** is considered acceptable (Knoben et al., 2019)

If metrics fall below these thresholds, consider:
1. Improving bathymetry resolution
2. Adjusting physics parameters (friction, breaking, whitecapping)
3. Refining boundary conditions
4. Increasing temporal/spatial resolution